## ① read_label_fixes

In [ ]:
%load_ext autoreload
%autoreload 2

from pill_detection.data import read_label_fixes

# 정제 목록에 AI Hub 줄도 있어서 Kaggle 줄만 읽습니다.
fixes = read_label_fixes("../../../data/label_fix/label_fixes.csv", source="kaggle")  # 현재 test.ipynb 파일 기준 경로
print(len(fixes))
fixes[:3]

In [ ]:
import pandas as pd

pd.DataFrame(fixes)

In [ ]:
help(read_label_fixes)

## ② load_annotations

In [ ]:
from pill_detection.data import load_annotations

# 노트북 위치(notebooks/eda/label_fix) 기준으로 세 단계 위가 레포 최상위 폴더입니다.
records = load_annotations("../../../data/raw/train_annotations")
print(len(records))
display(records[("K-003351-016262-018357_0_2_0_2_75_000_200.png", 18357)])
display(records[("K-003544-004543-012247-016548_0_2_0_2_75_000_200.png", 12247)])

In [ ]:
ann_df = pd.DataFrame(records.values())
print("JSON:", len(ann_df), "/ 이미지:", ann_df["file_name"].nunique(), "/ 클래스:", ann_df["category_id"].nunique())
print("annotation id:", ann_df["ann_id"].min(), "~", ann_df["ann_id"].max())
ann_df.head()

In [ ]:
ann_df[ann_df["file_name"] == "K-003351-016262-018357_0_2_0_2_75_000_200.png"]

In [ ]:
help(load_annotations)

## ③ check_label_fixes

In [ ]:
from pill_detection.data import check_label_fixes

plans = check_label_fixes(fixes, records, "../../../data/raw/train_images")
plan_df = pd.DataFrame(plans)
print(len(plans))
display(plan_df["action"].value_counts())

In [ ]:
with pd.option_context("display.max_colwidth", None): # 긴 경로가 잘리지 않도록, 이 블록 안에서만 칸 글자 수 제한(기본 50자)을 없앱니다.
    display(plan_df.loc[plan_df["action"] == "add", ["file_name", "category_id", "json_path", "drug_json_path", "image_json_path"]])

In [ ]:
with pd.option_context("display.max_colwidth", None):
    display(plan_df.loc[plan_df["action"] == "modify", ["file_name", "category_id", "json_path"]])

In [ ]:
help(check_label_fixes)

## ④ copy_annotations

In [ ]:
from pill_detection.data import copy_annotations

# 사본 annotation 폴더를 지우고 원본에서 다시 복사합니다. 이미지는 복사하지 않습니다.
n_copied = copy_annotations("../../../data/raw/train_annotations", "../../../data/processed/train_annotations")
n_copied

## ⑤ apply_label_fixes

In [ ]:
from pill_detection.data import apply_label_fixes

applied = apply_label_fixes(plans, records, "../../../data/raw/train_annotations", "../../../data/processed/train_annotations")
applied_df = pd.DataFrame(applied)
print(len(applied))
display(applied_df["action"].value_counts())

In [ ]:
# 긴 경로가 잘리지 않도록, 이 블록 안에서만 칸 글자 수 제한(기본 50자)을 없앱니다.
with pd.option_context("display.max_colwidth", None):
    display(applied_df)

In [ ]:
out_records = load_annotations("../../../data/processed/train_annotations")
print(len(out_records))
out_records[("K-003351-016262-018357_0_2_0_2_75_000_200.png", 18357)]

In [ ]:
key = ("K-003351-016262-018357_0_2_0_2_75_000_200.png", 18357)
print("원본:", records[key]["bbox"])
print("사본:", out_records[key]["bbox"])

## ⑥ write_label_changes

In [ ]:
from pill_detection.data import write_label_changes

# (data/label_fix)에 저장합니다.
path, changes = write_label_changes(records, out_records, "../../../data/label_fix/test_apply_label_fixes.csv")
print(path)
changes_df = pd.DataFrame(changes)
print(len(changes))
display(changes_df["action"].value_counts())

# 긴 파일명이 잘리지 않도록, 이 블록 안에서만 칸 글자 수 제한(기본 50자)을 없앱니다.
with pd.option_context("display.max_colwidth", None):
    display(changes_df)

In [ ]:
import io
import subprocess

# 통합 브랜치의 예전 수정 목록을 파일로 꺼내지 않고 git에서 바로 읽습니다.
old_text = subprocess.run(
    ["git", "show", "origin/eda/firt-eda-integrated:notebooks/eda/label_fix/jaeone_label_fixes.csv"],
    capture_output=True, encoding="utf-8-sig", check=True,
).stdout

# 두 CSV를 모두 글자 그대로 읽어서(빈칸도 빈 글자로) 같은 열끼리 비교합니다.
cols = ["file_name", "category_id", "bbox_before", "bbox_after", "area_before", "area_after"]
old_df = pd.read_csv(io.StringIO(old_text), dtype=str, keep_default_na=False)[cols]
new_df = pd.read_csv(path, dtype=str, keep_default_na=False)
# ⑧을 실행한 뒤의 기록에는 AI Hub 줄도 이어받아 있으므로 Kaggle 줄만 비교합니다.
if "source" in new_df.columns:
    new_df = new_df[new_df["source"] == "kaggle"]
new_df = new_df[cols]

merged = old_df.merge(new_df, on=["file_name", "category_id"], how="outer", suffixes=("_old", "_new"), indicator=True)
print("한쪽에만 있는 줄:", (merged["_merge"] != "both").sum())
print("값이 다른 열:", [c for c in cols[2:] if (merged[c + "_old"] != merged[c + "_new"]).any()] or "없음")

In [ ]:
from pathlib import Path

sorted(p.name for p in Path("../../../data/label_fix").iterdir())

## ⑦ make_fixed_annotations (①~⑥ 함수 결합)

In [ ]:
from pill_detection.data import make_fixed_annotations

result = make_fixed_annotations(
    fix_csv="../../../data/label_fix/label_fixes.csv",
    base_ann_dir="../../../data/raw/train_annotations",
    img_dir="../../../data/raw/train_images",
    out_ann_dir="../../../data/processed/train_annotations",
    change_csv="../../../data/label_fix/test_apply_label_fixes.csv",
)
print("기준 JSON:", result["n_base"], "/ 사본 JSON:", result["n_out"])
print("변경 기록:", result["change_csv"])
display(pd.DataFrame(result["changes"])["action"].value_counts())

In [ ]:
help(make_fixed_annotations)

## ⑧ make_train_dataset (Kaggle + AI Hub 학습 데이터 한 번에)

In [ ]:
from pill_detection.data import make_train_dataset

# Kaggle 사본, AI Hub 사본, 두 데이터의 변경 기록을 한 번에 만듭니다. (scripts/prepare_train_data.py가 부르는 함수)
# AI Hub 원본(data/external/aihub/raw)이 있어야 합니다. 없으면 레포 최상위에서 uv run scripts/prepare_train_data.py를 먼저 실행하세요.
# 몇 분 걸립니다.
train = make_train_dataset(
    fix_csv="../../../data/label_fix/label_fixes.csv",
    kaggle_ann_dir="../../../data/raw/train_annotations",
    kaggle_img_dir="../../../data/raw/train_images",
    aihub_ann_dir="../../../data/external/aihub/raw/train_annotations",
    aihub_img_dir="../../../data/external/aihub/raw/train_images",
    kaggle_out_ann_dir="../../../data/processed/train_annotations",
    aihub_out_dir="../../../data/processed/ai_hub",
    change_csv="../../../data/label_fix/test_apply_label_fixes.csv",
)
print("Kaggle JSON:", train["kaggle"]["n_base"], "→", train["kaggle"]["n_out"])
print("AI Hub 이미지:", train["aihub"]["n_images"], "/ JSON:", train["aihub"]["n_json"], "/ 이름을 고친 이미지:", len(train["aihub"]["renamed"]))
print("변경 기록:", train["change_csv"])
display(pd.DataFrame(train["changes"]).value_counts(["source", "action"]))

In [ ]:
help(make_train_dataset)